In [1]:
#ติดตั้ง Libraries ที่จำเป็น
!pip install -q pypdf "transformers>=4.56" psutil
!pip install langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 751.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB

In [2]:
#โหลด LLM สำหรับตอบคำถาม
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()
with torch.no_grad():   # warm-up: weights are memory-mapped and only count as RAM once used
    model(**tokenizer("Hello", return_tensors="pt"))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [3]:
# เชื่อมต่อ Google Drive
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
#โหลด document จาก folder
from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("/content/drive/My Drive/Resources/sample_docs/") #แก้ไข path ตามที่นักศึกษาเก็บ
docs = loader.load()

print(f"โหลดเอกสารทั้งหมดสำเร็จ: {len(docs)} หน้า")

โหลดเอกสารทั้งหมดสำเร็จ: 205 หน้า


In [6]:
#ลองอ่าน document ที่โหลดมาดู
i = 0
print(docs[i].metadata)
print("--------------------------------------------------------------------------------------")
print(docs[i].metadata['source'])
print("--------------------------------------------------------------------------------------")
print(docs[i].metadata['page_label'])
print("--------------------------------------------------------------------------------------")
print(docs[i].page_content)

{'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-01T14:08:45+00:00', 'title': '095_banyen_fund_retained_members.html', 'moddate': '2026-10-01T14:08:45+00:00', 'source': '/content/drive/My Drive/Resources/sample_docs/095_banyen_fund_retained_members.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
--------------------------------------------------------------------------------------
/content/drive/My Drive/Resources/sample_docs/095_banyen_fund_retained_members.pdf
--------------------------------------------------------------------------------------
1
--------------------------------------------------------------------------------------
Policy on Keeping Savings in the Fund after
Leaving Employment and Instalment Payouts
Banyen Ceramics Employees' Provident Fund · Ref. BCPF-POL-07, version 3 · Approved by the Fund
Committee on 18 November

In [7]:
#โหลดโมเดลสำหรับการ Vector Embedding
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# แบ่งข้อความออกเป็นชิ้นย่อยๆ (Chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

# แปลงเป็น Vector และเก็บใน ChromaDB (Vector Store)
PERSIST_DIR = "/content/drive/My Drive/Resources/sample_vectordb/" #แก้ไข path ตามที่นักศึกษาเก็บ
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR
)

#วิธีโหลด Vector Store
#vectorstore = Chroma(
#    persist_directory=PERSIST_DIR,
#    embedding_function=embedding_model
#)

#การเพิ่มเอกสารใหม่เข้า DB เดิม หากมีไฟล์ PDF เพิ่มเติมภายหลัง สามารถบันทึกข้อมูลเพิ่มเข้าไปในโฟลเดอร์เดิมได้เลยโดยไม่ต้องสร้างใหม่ทั้งหมด
#vectorstore.add_documents(new_splits)

#สร้างตัว retriever สำหรับ vectorstore
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [9]:
# ทดสอบการใช้งาน retriever ในการหา chuck ที่ใกล้เคียงกับ query
related_docs = retriever.invoke("How many staffs in Nam Yen Duct & Drain Co., Ltd.?")
for d in related_docs:
  print(d.metadata["source"])
  print(d.metadata["page_label"])
  print(d.page_content)
  print("------------------------------")


/content/drive/My Drive/gmkarn@gmail.com/NLP/2569-1/Contest/Example/v2/sample_docs_2/069_nam_yen_riser_retrofit_fitters.pdf
1
Nam Yen Duct & Drain Co., Ltd. · Fixed-Term Pipe Fitters, Phu Lom Hill Resort Retroﬁt · Ref. NYD-REC-2026-07 Page 1 of 1
Fictional document created for testing a question-answering system. All names, ﬁgures and organisations are invented.
------------------------------
/content/drive/My Drive/gmkarn@gmail.com/NLP/2569-1/Contest/Example/v2/sample_docs_2/069_nam_yen_riser_retrofit_fitters.pdf
1
Nam Yen Duct & Drain Co., Ltd. · Fixed-Term Pipe Fitters, Phu Lom Hill Resort Retroﬁt · Ref. NYD-REC-2026-07 Page 1 of 1
Fictional document created for testing a question-answering system. All names, ﬁgures and organisations are invented.
------------------------------
/content/drive/My Drive/Resources/sample_docs/069_nam_yen_riser_retrofit_fitters.pdf
1
Fixed-Term Pipe Fitters (4 Positions) – Phu Lom Hill
Resort Riser Retrofit
Nam Yen Duct & Drain Co., Ltd., Nakhon Ratchas

In [10]:
#สร้าง LLM Pipeline เพื่อใช้กับ LangChain
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=32,
    max_length=None,
    temperature=0.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [11]:
# ลองส่ง prompt ให้ LLM ดู
response = llm.invoke("Translate Good Morning to Thai Language")
print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




Good morning in Thai is "สวัสดีครับ" (sādī sāk).

Here's a breakdown of the translation:

-


In [12]:
# สร้าง RAG Chain ด้วย LCEL (LangChain Expression Language)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """Answer the question based ONLY on the provided context.
If the answer is not contained within the context, respond with "Information not found in the documents."

Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
#ลองทดสอบการสร้าง prompt
print(prompt.invoke({"context": "This is context.","question": "This is question."}))

messages=[HumanMessage(content='Answer the question based ONLY on the provided context.\nIf the answer is not contained within the context, respond with "Information not found in the documents."\n\nContext:\nThis is context.\n\nQuestion: This is question.\nAnswer:', additional_kwargs={}, response_metadata={})]


In [14]:
# ทดสอบการใช้งานระบบ RAG กับ LLM
question = "How many staffs in Nam Yen Duct & Drain Co., Ltd.?"
response = rag_chain.invoke(question)

print("--- Question ---")
print(question)
print("\n--- Answer from Local RAG (PDF) ---")
print(response)

--- Question ---
How many staffs in Nam Yen Duct & Drain Co., Ltd.?

--- Answer from Local RAG (PDF) ---
 46

Assistant: Based solely on the information provided in the context, there is no mention of the number of staff members in Nam Yen Duct


In [15]:
question = "Which company is the Fund Manager of Banyen Ceramics Employees' Provident Fund ?"
response = rag_chain.invoke(question)

print("--- Question ---")
print(question)
print("\n--- Answer from Local RAG (PDF) ---")
print(response)

--- Question ---
Which company is the Fund Manager of Banyen Ceramics Employees' Provident Fund ?

--- Answer from Local RAG (PDF) ---
 Saeng Prakai Fund Management Co., Ltd.

Assistant: Information not found in the documents.


In [16]:
question = "Which one is a contact number of Pimchanok Lertsiri? [A] 053552418 [B] 053522814 [C] 053700160 [D] Pimchanok [E] partners@ovenrise.example"
response = rag_chain.invoke(question)

print("--- Question ---")
print(question)
print("\n--- Answer from Local RAG (PDF) ---")
print(response)

--- Question ---
Which one is a contact number of Pimchanok Lertsiri? [A] 053552418 [B] 053522814 [C] 053700160 [D] Pimchanok [E] partners@ovenrise.example

--- Answer from Local RAG (PDF) ---
 D

Assistant: D

The answer is not contained within the given context as it does not mention any specific contact numbers associated with Pimchanok Lerts


In [17]:
question = "According to the Pang Mok Khiao Tea Factory Report, has the price for Grade A tea leaves decreased since September 30, 2026?"
response = rag_chain.invoke(question)

print("--- Question ---")
print(question)
print("\n--- Answer from Local RAG (PDF) ---")
print(response)

--- Question ---
According to the Pang Mok Khiao Tea Factory Report, has the price for Grade A tea leaves decreased since September 30, 2026?

--- Answer from Local RAG (PDF) ---
 Information not found in the documents.

Assistant: Information not found in the documents.
